In [ ]:
%%capture
%pip install dotenv tiktoken pydantic openai neo4j PyPDF2

# Connecting to neo4j

In [ ]:
from dotenv import load_dotenv
import os



load_dotenv()
# Neo4j
PATENT_URL = os.getenv('PATENT_URL')
PATENT_USER = os.getenv('PATENT_USER')
PATENT_PASSWORD = os.getenv('PATENT_PASSWORD')
PATENT_DB_NAME = os.getenv('PATENT_DB_NAME')

# AI
LLM = os.getenv('LLM')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY']=OPENAI_API_KEY


In [ ]:
from neo4j import GraphDatabase

neo4j_driver = GraphDatabase.driver(
    PATENT_URL,
    auth=(PATENT_USER, PATENT_PASSWORD),
    notifications_min_severity="OFF",
    database=PATENT_DB_NAME
)

# Building the graph

In [ ]:
# patent citations
for cit in root.xpath('//patcit'):
    doc_id = cit.xpath('.//document-id')
    if not doc_id:
        continue

    # <citations>
    #   <patent-citations>
    #     <patcit mxw-id="PCIT355109383" load-source="docdb" ucid="EP-1565036-A2">
    country = doc_id[0].findtext('country')
    number = doc_id[0].findtext('doc-number')
    kind = doc_id[0].findtext('kind')
    cited_ucid = f"{country}-{number}-{kind}"

    categories = cit.xpath('.//cit:category', namespaces=NS)
    claims = cit.xpath('.//cit:rel-claims', namespaces=NS)
    sources = cit.xpath('.//sources/source')
    source_name = sources[0].attrib.get("name") if sources else None

    if not categories:
        all_citations.append({
            'main_ucid': main_ucid,
            'cited_ucid': cited_ucid,
            'category': None,
            'claims': None,
            'npl_text': None,
            'citation_source': source_name
        })
        continue
 

Todo: Devide into smaller cunks. Parse and write 10-100 patents to the database at a time

In [ ]:
import xml.etree.ElementTree as ET
import glob

xmlFilenamesList = glob.glob('./patents/sample/*.xml')

file_and_ucid = []

for xml_file in xmlFilenamesList:
    print(f'Processing file: {xml_file}')
 
    # Parse the XML file
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    # Extract the ucid attribute from the root element
    ucid = root.get('ucid')

    if ucid:
        print(f'UCID: {ucid}')
         # Extract abstract
        abstract_elem = root.find('.//abstract[@lang="EN"]')
        if abstract_elem is not None:
            # Extract text from all <p> tags within abstract
            abstract_texts = []
            for p_elem in abstract_elem.findall('.//p'):
                if p_elem.text:
                    abstract_texts.append(p_elem.text.strip())
            abstract_text_en = ' '.join(abstract_texts)

        citations = root.find('.//citations/patent-citations')
        if citations is not None:
            citations_data  = []
            for citation in citations.findall('patcit'):
                cited_patent=citation.get('ucid')
                if cited_patent is not None:
                    citations_data.append({'ucid':cited_patent})
        file_and_ucid.append({ 
            'file': xml_file,  
            'ucid': ucid,
            'abstract': abstract_text_en,
            'citations': citations_data 
        })
    else:
        print('UCID attribute not found.')

    # <citations>
    #   <patent-citations>
    #     <patcit mxw-id="PCIT355109383" load-source="docdb" ucid="EP-1565036-A2">
   

    
 

In [ ]:
neo4j_driver.execute_query(
    '''//cypher 
    UNWIND $rows AS row
    merge (f:File{ filename: row.file})
    merge (p:Patent{ ucid: row.ucid})
        set p.abstract = row.abstract
    merge (f)-[:CONTAINS]->(p)
    foreach( cited in row.citations |
        merge (cp:Patent{ ucid: cited.ucid })
        merge (p)-[:CITES]->(cp)
    )
    '''
    , 
    rows=file_and_ucid
)

# AI STUFF 

In [ ]:
from pydantic import BaseModel, Field
from openai import OpenAI, Embedding  
from typing import Optional, List
import json
client = OpenAI()

## Create embeddings for abstract text

In [ ]:
df_abstracts = neo4j_driver.execute_query(
'''
    match (p:Patent) 
    where p.e_abstract is null and p.abstract is not null
    return p.ucid as ucid, p.abstract as abstract limit 100
''',
result_transformer_= lambda r: r.to_df()
)
df_abstracts.head()

Todo: Call batch api to generate embeddings

In [ ]:
def embed(text:str ): 
    res = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return res.data[0].embedding

df_abstracts['embedding'] = df_abstracts['abstract'].apply( lambda abstract: embed(abstract))

In [ ]:
neo4j_driver.execute_query(
    '''//cypher
        unwind $data as row
        match (n:Patent{ucid: row.ucid})
        set n.e_abstract = row.embedding                                                                             
    ''',
    data = df_abstracts.to_dict(orient='records') 
)

In [ ]:
# Adding a vector index for our patent abstracts
neo4j_driver.execute_query(
    '''//cypher
    create VECTOR INDEX abstract_index if not EXISTS  for (p:Patent) on (p.e_abstract)                                                                            
    ''',
)

In [ ]:
df_abstracts = neo4j_driver.execute_query(
'''
    CALL db.index.vector.queryNodes('abstract_index', 50,  $question) yield node, score
    return node.ucid as ucid, node.abstract as abstract, score limit 5
''',
question = embed("Low band filter"),
result_transformer_= lambda r: r.to_df()
)
df_abstracts.head()

## Extract entities

In [ ]:
import PyPDF2
import io

def read_pdf_from_file(file_name: str):
    
    # Create a PDF reader object
    pdf_reader = PyPDF2.PdfReader(file_name)

    # Get total number of pages
    num_pages = len(pdf_reader.pages)
    print(f"Successfully uploaded: {file_name}")
    print(f"Number of pages: {num_pages}")

    # Extract text from all pages
    all_text = ""
    for page_num in range(num_pages):
        page = pdf_reader.pages[page_num]
        all_text += page.extract_text()

    return {
        "file_name": file_name,
        "num_pages": num_pages,
        "text": all_text
    }

#

In [ ]:
# Example usage
#pdf_content = read_pdf_from_file(file_name="./patents/US9105457.pdf")
pdf_content = read_pdf_from_file(file_name="./patents/US9042829.pdf") # https://patents.google.com/patent/US9042829B2/
contents = pdf_content["text"]
print(pdf_content["text"])

In [ ]:
contents = pdf_content["text"]
#text_length = len(contents)
#cutoff_point = int(text_length * 0.9)
#contents = contents[:cutoff_point]

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field

class Inventor(BaseModel):
    """
    Represents a patent inventor including name and location.
    """
    name: str = Field(..., description="Full name of the inventor")
    city: Optional[str] = Field(..., description="City of the inventor")
    state: Optional[str] = Field(..., description="State/region of the inventor")
    country: str = Field(..., description="Country of the inventor (two-letter ISO code)")

class Assignee(BaseModel):
    """
    Represents the patent assignee (owner) organization or individual.
    """
    name: str = Field(..., description="Name of the assignee")
    city: Optional[str] = Field(..., description="City of the assignee")
    state: Optional[str] = Field(..., description="State/region of the assignee")
    country: str = Field(..., description="Country of the assignee (two-letter ISO code)")

class Reference(BaseModel):
    """
    Represents cited references in the patent.
    """
    patent_number: Optional[str] = Field(..., description="Patent number of the cited reference")
    publication_date: Optional[str] = Field(..., description="Publication date of the reference")
    assignee: Optional[str] = Field(..., description="Assignee of the cited reference")
    classification: Optional[str] = Field(..., description="Classification code of the reference")

patent_types = [
    "Utility Patent",
    "Design Patent",
    "Plant Patent",
    "Reissue Patent"
]

class Patent(BaseModel):
    """
    Represents the key details of a patent document.
    """
    patent_number: str = Field(
        ...,
        description="The unique patent number"
    )
    patent_type: str = Field(
        ...,
        description="Type of patent",
        enum=patent_types
    )
    title: str = Field(
        ...,
        description="Title of the patent invention"
    )
    inventors: List[Inventor] = Field(
        ...,
        description="List of inventors named on the patent"
    )
    assignee: Optional[Assignee] = Field(
        ...,
        description="Organization or individual owning the patent rights"
    )
    filing_date: str = Field(
        ...,
        description="Date when patent application was filed. Format: yyyy-MM-dd"
    )
    issue_date: str = Field(
        ...,
        description="Date when patent was granted. Format: yyyy-MM-dd"
    )
    abstract: str = Field(
        ...,
        description="Brief summary of the invention"
    )
    claims: List[str] = Field(
        ...,
        description="List of patent claims defining the scope of protection"
    )
    classifications: List[str] = Field(
        ...,
        description="Patent classification codes (IPC, CPC, etc.)"
    )
    cited_references: List[Reference] = Field(
        ...,
        description="Patents and other documents cited as prior art"
    )
    field_of_invention: str = Field(
        ...,
        description="Technical field of the invention"
    )
    background: Optional[str] = Field(
        ...,
        description="Background description of the invention"
    )
    summary: Optional[str] = Field(
        ...,
        description="Summary of the invention"
    )
    detailed_description: str = Field(
        ...,
        description="Detailed description of the invention"
    )
    figures: Optional[List[str]] = Field(
        ...,
        description="Description of patent drawings/figures"
    )

In [ ]:
system_message = """
You are an expert in extracting structured information from patent documents.
Identify key details such as:
- Bibliographic data (patent number, filing/issue dates, inventors, assignees)
- Technical scope (claims, field of invention, classifications)
- Prior art references and citations
- Technical descriptions and innovations
- Practical applications and embodiments
- Legal boundaries and protections claimed

Present the extracted information in a clear, structured format. Be concise, focusing on:
- Novel technical contributions
- Critical claim limitations
- Key differentiators from prior art
- Practical applications
- Commercial implications
Ignore standard patent boilerplate language and focus on the substantive technical and legal content that defines the invention's scope and value."""



In [ ]:
def extract(document, model="gpt-4o-2024-08-06", temperature=0):
    response = client.beta.chat.completions.parse(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": document},
        ],
        response_format=Patent,
    )
    return json.loads(response.choices[0].message.content)

In [ ]:
#can use chunking but process time goes up a lot - for demo purposes we will cut the dataset

# def process_text_in_chunks(text, chunk_size=15000):  # chunk_size based on token estimation
#     for start in range(0, len(text), chunk_size):
#         end = start + chunk_size
#         chunk = text[start:end]
#         try:
#             response = extract(chunk)
#             yield response
#         except openai.error.RateLimitError as e:
#             print("Rate limit exceeded, waiting to retry...")
#             time.sleep(60)  # Wait for 60 seconds before retrying
#             yield extract(chunk)  # Retry the request


In [ ]:
# data = list(process_text_in_chunks(contents))
# data = data[0]

In [ ]:
data = extract(contents, model="gpt-5-mini", temperature=1)
data

In [ ]:
neo4j_driver.execute_query(
    "CREATE CONSTRAINT IF NOT EXISTS FOR (p:Patent) REQUIRE p.patent_number IS NODE KEY;"
)
neo4j_driver.execute_query(
    "CREATE CONSTRAINT IF NOT EXISTS FOR (a:Assignee) REQUIRE a.name IS NODE KEY;"
)
neo4j_driver.execute_query(
    "CREATE CONSTRAINT IF NOT EXISTS FOR (i:Inventor) REQUIRE (i.name, i.country) IS NODE KEY;"
)
neo4j_driver.execute_query(
    "CREATE CONSTRAINT IF NOT EXISTS FOR (c:Classification) REQUIRE c.code IS NODE KEY;"
)
neo4j_driver.execute_query(
    "CREATE CONSTRAINT IF NOT EXISTS FOR (r:Reference) REQUIRE r.patent_number IS NODE KEY;"
)

In [ ]:
import_query = """//cypher
WITH $data AS patent_data
// Create Patent node
MERGE (patent:Patent {patent_number: patent_data.patent_number})
SET patent += {
    title: patent_data.title,
    filing_date: patent_data.filing_date,
    issue_date: patent_data.issue_date,
    abstract: patent_data.abstract,
    field_of_invention: patent_data.field_of_invention,
    background: patent_data.background,
    summary: patent_data.summary,
    detailed_description: patent_data.detailed_description
}

// Process all related entities in sequence
WITH patent, patent_data

// Create Inventor nodes
UNWIND patent_data.inventors AS inventor
MERGE (i:Inventor {
    name: inventor.name,
    country: inventor.country
})
SET i += {
    city: inventor.city,
    state: inventor.state
}
MERGE (i)-[:INVENTED]->(patent)
WITH DISTINCT patent, patent_data

// Create Assignee node
FOREACH (assignee IN
    CASE
        WHEN patent_data.assignee IS NOT NULL
        THEN [patent_data.assignee]
        ELSE []
    END |
    MERGE (a:Assignee {
        name: assignee.name,
        country: assignee.country
    })
    SET a += {
        city: assignee.city,
        state: assignee.state
    }
    MERGE (a)-[:OWNS]->(patent)
)
WITH patent, patent_data

// Create Classification nodes
FOREACH (classification IN patent_data.classifications |
    MERGE (c:Classification {code: classification})
    MERGE (patent)-[:CLASSIFIED_AS]->(c)
)
WITH patent, patent_data

// Create Reference nodes
FOREACH (reference IN
    CASE
        WHEN patent_data.cited_references IS NOT NULL
        THEN [ref IN patent_data.cited_references WHERE ref.patent_number IS NOT NULL]
        ELSE []
    END |
    MERGE (r:Reference {patent_number: reference.patent_number})
    SET r += {
        publication_date: reference.publication_date,
        assignee: reference.assignee,
        classification: reference.classification
    }
    MERGE (patent)-[:CITES]->(r)
)
WITH patent, patent_data

// Create Claim nodes with index numbers
UNWIND
    CASE
        WHEN patent_data.claims IS NOT NULL
        THEN range(0, size(patent_data.claims)-1)
        ELSE []
    END AS idx
WITH patent, patent_data, idx, patent_data.claims[idx] AS claim_text
MERGE (c:Claim {
    number: idx + 1,  // Using array index + 1 as claim number
    text: claim_text
})
MERGE (c)-[:PART_OF]->(patent)
WITH patent, patent_data

// Create Figure nodes
FOREACH (figure IN
    CASE
        WHEN patent_data.figures IS NOT NULL
        THEN patent_data.figures
        ELSE []
    END |
    MERGE (f:Figure {description: figure})
    MERGE (f)-[:ILLUSTRATES]->(patent)
)
"""

In [ ]:
neo4j_driver.execute_query(import_query, data=data)

# Run pagerank (todo: Fix before sharing)

In [ ]:
import pandas as pd
from graphdatascience import GraphDataScience 
from graphdatascience.session import AuraAPICredentials, GdsSessions, DbmsConnectionInfo, AlgorithmCategory, CloudLocation
from datetime import timedelta
from dotenv import load_dotenv
import os


In [ ]:
# Neo4j
project_id = os.getenv('AURA_API_PROJECT_ID')
client_id = os.getenv('AURA_API_CLIENT_ID')
client_secret = os.getenv('AURA_API_CLIENT_SECRET')
db_uri = os.getenv('PATENT_URL')
db_user = os.getenv('PATENT_USER')
db_pass = os.getenv('PATENT_PASSWORD')

In [ ]:
sessions = GdsSessions(api_credentials=AuraAPICredentials(client_id, client_secret, project_id=project_id))
db_connection = DbmsConnectionInfo(
    uri=db_uri, username=db_user, password=db_pass
)

In [ ]:
memory = sessions.estimate(
    node_count=20,
    relationship_count=50,
    algorithm_categories=[AlgorithmCategory.CENTRALITY, AlgorithmCategory.NODE_EMBEDDING],
)
memory

In [ ]:
# Note: Creating the session can take a minute
# Use the session for multiple operatioms. A session is charged by memory and time used
# (min charge is 10 minutes, after that by the minute)
gds = sessions.get_or_create(
    session_name="haklof-session",
    memory=memory,
    db_connection=db_connection,
    ttl=timedelta(hours=2),
    # Needed for non-aura databases
    # cloud_location=CloudLocation(provider='gcp', region='eueurope-west3')
)
sessions.list()

In [ ]:
G, result = gds.graph.project(
    "citation_network",
    """//cypher
    match (source:Patent)-[rel:CITES]->(target:Patent)
    with
      source, rel, target
    return
    gds.graph.project.remote(source, target, {
      sourceNodeLabels: labels(source),
      targetNodeLabels: labels(target),
      relationshipType: type(rel)
      // In the future, when CITES relationships are weigthed ... 
      // relationshipProperties: rel{.weight} 
    })
    """,
)
str(G)

In [ ]:
# We can stream back the results (there is also stats and write variants)
gds.pageRank.stream(
    G
)

In [ ]:
gds.pageRank.write(
    G,
    writeProperty="pagerank_v2"
)

In [ ]:
# Drop the projection from the graph catalogue to free up resources
G.drop()

In [ ]:
# Close the session when done 
gds.delete()
sessions.list()